# 00 — The real NOSIBLE API, as a reference point

Blog: [The Road to Cybernaut-1](https://nosible.com/blog/the-road-to-cybernaut-1)

This notebook is **not** part of `cybernaut-mini`. It calls the real, hosted NOSIBLE
search API. It is kept because the shape of that API is itself evidence for what the
blog describes — and it gives us a ground truth to compare our replica against.

## Why this matters for the replica

Look at the request body below. It carries an **`instruction`** field alongside the
**`question`**. That is not decoration — it is the public surface of blog **Stage 4,
"Instruction Tuning and Embedding"**:

> "we generate an appropriate instruction for the question and submit it, along with
> the expansions, to `multilingual-e5-large-instruct`. E5 is an open-source,
> instruction-tuned embedding model from Microsoft Research. It is designed to align
> vectors with natural language instructions."

The blog claims optimizing this instruction yields "a free 1-5% improvement in search
precision and recall". The API exposing `instruction` as a caller-controlled parameter
is consistent with that claim, and is why our replica treats the instruction template
as a **tunable knob** rather than a constant (see `query/s4_embed/`).

Note also `n_results: 10` — and recall the blog's design principle #2, *recall over
precision*: "People need precision; AIs need recall." The worked examples in the post
return **100** results.

## Credentials

This notebook reads `NOSIBLE_API_KEY` from the environment. **Never paste a key into a
notebook cell** — notebooks serialize their source to disk and get committed. An
earlier version of this file had a live key hardcoded in three cells.

```bash
export NOSIBLE_API_KEY="nos_sk_..."
```

If the variable is unset, every cell below degrades to a no-op with an explanatory
message rather than failing — this notebook is documentation first, and must remain
readable without an account.

In [ ]:
import json
import os

API_KEY = os.environ.get("NOSIBLE_API_KEY")
ENDPOINT = "https://www.nosible.ai/search/v1/fast-search"

# Live calls cost money and need the network, and this notebook is executed by the
# test suite on every `make check`. The devcontainer makes NOSIBLE_API_KEY ambient
# (it injects .env), so a key alone must NOT be enough to fire a request — opting in
# is explicit and deliberate:
#
#     CYBERNAUT_LIVE_API=1 jupyter lab      # or set it in the notebook UI
#
LIVE = bool(API_KEY) and os.environ.get("CYBERNAUT_LIVE_API") == "1"

if LIVE:
    print(f"LIVE mode: NOSIBLE_API_KEY found (ends ...{API_KEY[-4:]})")
elif API_KEY:
    print("Key present, but CYBERNAUT_LIVE_API is not set to 1 — live cells are skipped.")
    print("This is the default so automated runs never spend credit.")
else:
    print("NOSIBLE_API_KEY is not set — live cells are skipped.")
print("The notebook is fully readable as documentation either way.")

## The request shape

Shown as data rather than executed, so the structure is legible even offline.
Each field is annotated with the blog stage it corresponds to.

In [ ]:
REQUEST_SHAPE = {
    "page": {
        # Stage 4 — Instruction Tuning and Embedding.
        # The instruction is prepended to the query before embedding, steering the
        # vector. This is the knob the blog says is worth 1-5% precision/recall.
        "instruction": "Retrieve semantically similar text.",
        # Stages 1-3 — the raw question, which the engine detects the language of,
        # tokenizes, and extracts search intents from.
        "question": "Why did Argus Filch confiscate the Marauder's Map?",
        # Stage 8 — how many results to reduce to. The blog's design principle is
        # "recall over precision": AIs have massive context windows, so ask for many.
        "n_results": 10,
    }
}

print(json.dumps(REQUEST_SHAPE, indent=2))

In [ ]:
def fast_search(question: str, instruction: str, n_results: int = 10) -> dict | None:
    """Call the hosted NOSIBLE API. Returns None unless live calls are opted into."""
    if not LIVE:
        print("Skipped: live calls are off (set CYBERNAUT_LIVE_API=1 with a valid key).")
        return None

    import requests

    response = requests.post(
        ENDPOINT,
        headers={"Content-Type": "application/json", "api_key": API_KEY},
        json={
            "page": {
                "instruction": instruction,
                "question": question,
                "n_results": n_results,
            }
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


result = fast_search(
    question="Why did Argus Filch confiscate the Marauder's Map?",
    instruction="Retrieve semantically similar text.",
)

if result:
    for hit in result["response"]:
        print(hit["title"])

## Experiment: does the instruction actually change the results?

This is the blog's Stage 4 claim, testable directly against the live API. If the
instruction is a real parameter and not a no-op, two different instructions over the
*same* question should return measurably different result sets.

We run the blog's own worked-example query so the comparison is on their turf.

In [ ]:
BLOG_QUESTION = (
    "What lessons from bacteria and yeast actually translate into "
    "safer gene-editing medicines?"
)

# A generic instruction vs. the evolved template quoted in the blog.
INSTRUCTIONS = {
    "generic": "Retrieve semantically similar text.",
    "blog_evolved": (
        "Given a question, please retrieve any relevant English Headlines, Leads, "
        "Passages, and Source URLs that focus on the same named entities as the "
        "question, and provide substantive answers to the question."
    ),
}

titles: dict[str, list[str]] = {}
for name, instruction in INSTRUCTIONS.items():
    payload = fast_search(BLOG_QUESTION, instruction, n_results=10)
    if payload is None:
        break
    titles[name] = [hit["title"] for hit in payload["response"]]
    print(f"\n--- {name} ---")
    for title in titles[name]:
        print(" ", title)

if len(titles) == 2:
    a, b = (set(v) for v in titles.values())
    overlap = len(a & b) / len(a | b) if (a | b) else 0.0
    print(f"\nJaccard overlap between the two result sets: {overlap:.2f}")
    print(
        "Overlap well below 1.0 means the instruction genuinely steers retrieval, "
        "which is what blog Stage 4 claims."
    )

## What this tells us to build

| Observation from the live API | Consequence for `cybernaut-mini` |
|---|---|
| `instruction` is a caller-supplied parameter | Stage 4 needs a **template registry**, not a hardcoded string — `query/s4_embed/` |
| `question` is passed raw | Language detection, tokenization and intent extraction happen **server-side** — stages 1-3 are ours to build |
| `n_results` defaults generously | Reduce phase must scale to ~100 hits, not 10 — blog principle "recall over precision" |
| Response items carry a `title` | Snippets and titles are the product surface — see Stage 8 snippet construction |

**Alternatives to calling this API at all:** we could have treated the blog as the
sole specification and never touched the hosted service. We keep this notebook because
a reachable reference implementation is a cheap oracle — when our replica behaves
oddly, it is useful to know how the real thing responds to the same input. It is *not*
used for evaluation: our metrics come from real human relevance judgments (SciFact
qrels), not from agreement with a black box.

---

## Our replica, loaded from the catalog

Everything above describes the *hosted* system. This section runs the same shape of
query against **our** index — read through the Kedro catalog, exactly as the pipelines
read it — so the comparison is a calculation rather than an assertion.

```mermaid
flowchart LR
    subgraph HOSTED["hosted NOSIBLE"]
        A["8-stage pipeline<br/>250k shards"]
    end
    subgraph OURS["cybernaut-mini"]
        C[("catalog<br/>shard_index")] --> D["retrieve()<br/>lexical · dense · hybrid"]
    end
    Q(["same question"]) --> A
    Q --> C
    A -.->|"titles"| CMP{"compare"}
    D -.->|"titles"| CMP
```

In [ ]:
from cybernaut_mini.config import RRFConfig
from cybernaut_mini.notebook import ensure_sample_index, kedro_catalog
from cybernaut_mini.retrieval import provider_from_meta, retrieve
from cybernaut_mini.text import TextProcessor

ensure_sample_index()
catalog = kedro_catalog(index_path="artifacts/sample")

index = catalog.load("shard_index")
provider = provider_from_meta(index.meta, offline=True)
processor = TextProcessor(use_spacy=False)

print(f"replica index: {index.meta.n_documents} documents in {index.meta.n_shards} shards")
print(f"embedder     : {provider.identifier}")

### The blog's own question, against our corpus

The hosted API answers this from a web-scale corpus. Ours has 63 documents, so the
point is emphatically *not* result quality — it is that the same question flows
through the same stage shape and comes back ranked.

In [ ]:
BLOG_QUESTION_LOCAL = "gene editing techniques for safer medicines"

print(f"question: {BLOG_QUESTION_LOCAL!r}\n")
for mode in ("lexical", "dense", "hybrid"):
    hits = retrieve(
        index, BLOG_QUESTION_LOCAL, mode=mode, processor=processor,
        provider=provider, rrf_config=RRFConfig(), top_k=5,
    )
    print(f"--- {mode} ---")
    for hit in hits:
        print(f"  {hit.rank}. [{hit.score:.4f}] shard={hit.shard_id}  {hit.document.title[:62]}")
    print()

### Instruction steering — what we do *not* have

Blog stage 4 prepends an instruction to the query before embedding, and the live
experiment earlier in this notebook shows it genuinely reorders results.

Our replica has no instruction template registry: `retrieve()` takes a question and
nothing else. The cell below makes the gap explicit rather than leaving it implied —
two different phrasings of the same information need, and what our index does with
them.

In [ ]:
PHRASINGS = {
    "plain": "gene editing techniques for safer medicines",
    "instructed": (
        "Given a question, please retrieve any relevant English Headlines, Leads, "
        "Passages, and Source URLs: gene editing techniques for safer medicines"
    ),
}

ranked: dict[str, list[str]] = {}
for name, text in PHRASINGS.items():
    hits = retrieve(
        index, text, mode="hybrid", processor=processor,
        provider=provider, rrf_config=RRFConfig(), top_k=5,
    )
    ranked[name] = [hit.document.id for hit in hits]
    print(f"{name:<11}: {ranked[name]}")

a, b = (set(v) for v in ranked.values())
jaccard = len(a & b) / len(a | b) if (a | b) else 0.0
print()
print(f"Jaccard overlap between the two phrasings: {jaccard:.2f}")
print()
print("The hosted API treats the instruction as a separate, embedded field. We simply")
print("concatenate it into the query, so the extra words act as noise rather than")
print("steering. That is the concrete shape of the missing stage-4 template registry.")

### Comparing against the live API

Runs only when `CYBERNAUT_LIVE_API=1` and a key are both present. Otherwise it reports
that it was skipped — which is the default, including in CI.

In [ ]:
if LIVE:
    payload = fast_search(
        BLOG_QUESTION,
        INSTRUCTIONS["blog_evolved"],
        n_results=10,
    )
    if payload:
        hosted_titles = [hit["title"] for hit in payload["response"]]
        print(f"hosted returned {len(hosted_titles)} results:")
        for title in hosted_titles:
            print("  ", title)
        print()
        print("Our corpus shares no documents with the live web index, so title overlap")
        print("is expected to be zero. The comparable things are the *stage shape* and")
        print("the response contract, not the documents themselves.")
else:
    print("Skipped — live comparison needs CYBERNAUT_LIVE_API=1 and NOSIBLE_API_KEY.")
    print()
    print("What a live run would add: the hosted result set for the same question, to")
    print("compare response contract and ranking behaviour against our replica above.")

## Where the replica stands

| Blog stage | Hosted API | `cybernaut-mini` |
|---|---|---|
| 1-3 language detect / tokenize / intent | server-side, opaque | tokenization built; language detect and intent not |
| 4 instruction tuning + embedding | `instruction` parameter, demonstrably steers | **not built** — no template registry |
| 5-6 shard selection + reranking | 250k shards | built, flat over 8 shards ([`02_shard_anatomy`](02_shard_anatomy.ipynb)) |
| 7 query expansion | shard-based | built (heuristic) |
| 8 map-reduce retrieval | across selected shards | built ([`03_retrieval_evaluation`](03_retrieval_evaluation.ipynb)) |

Continue with [`01_corpus_ingest.ipynb`](01_corpus_ingest.ipynb) for how documents get
into the index in the first place.